In [ ]:
%%capture

import warnings
warnings.filterwarnings("ignore")

import altair as alt
import branca.colormap as cm
import folium
import gcsfs
import pandas as pd

import calitp_data_analysis.magics
import gt_extras as gte

from great_tables import GT, md

import chart_utils_for_operators as chart_utils
import prep_operator_data
import report_utils
import _color_palette
PREDICTIONS_GCS = "gs://calitp-analytics-data/data-analyses/rt_predictions/"

alt.data_transformers.enable("vegafusion")

one_month = pd.to_datetime("2026-02-01")
date_format = "%b %Y" # gtfs_digest/_new_operator_report_utils.py
one_month_formatted = one_month.strftime(date_format)

In [ ]:
operator_cols = ["tu_name", "day_type"]

schedule_cols = [
    "daily_trips", "daily_service_hours", "n_routes", "n_shapes",
    "n_stops", "num_stop_times", "daily_arrivals", 'n_days_schedule_and_rt'
]

tu_cols = ['tu_messages_per_minute', 'n_tu_trips', 'pct_tu_trips', 'pct_tu_routes'] #'daily_tu_trips',

# split prediction cols into 2 separate tables so comparison is clearer
tu_prediction_cols1 = ["bus_catch_likelihood", "pct_tu_complete_minutes", "p50", "n_predictions"]
tu_prediction_cols2 = [
    "p25", "p75", "iqr",
    "prediction_padding_minutes", "avg_prediction_spread_minutes"
]

In [ ]:
equans = "Marin EQUANS Trip Update"
swiftly = "Marin Swiftly Trip Update"
mtc = "Bay Area 511 Marin TripUpdates"

df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
).pipe(prep_operator_data.merge_in_operator_percentiles)

In [ ]:
# Set variables for color bars used across maps, route dropdown, and great tables
PREDICTION_ERROR_COLORS =list(_color_palette.PREDICTION_ERROR_COLOR_PALETTE.values())
PREDICTION_ERROR_INDEX = [-5, -3, -1, 1, 3, 5]
PREDICTION_ERROR_LEGEND_CAPTION = "minutes (negative = late; positive = early)"

POS_BAR_COLOR = _color_palette.get_color("blueberry")
NEG_BAR_COLOR = _color_palette.get_color("vivid_cerise")

## Schedule + RT Summary Stats

In [ ]:
schedule_table = (
    GT(df[["schedule_name"] + operator_cols + schedule_cols])
    .cols_label(
        daily_trips = "Daily Trips",
        daily_service_hours = "Daily Service Hours",
        n_routes = "# Routes",
        n_shapes = "# Shapes",
        n_stops = "# Stops",
        num_stop_times = "Total Scheduled Arrivals",
        daily_arrivals = "Daily Scheduled Arrivals",    
        n_days_schedule_and_rt = "# days with both RT",
    ).fmt_integer(
        columns = [
            "daily_trips", "n_routes", "n_shapes", "n_stops", 
            "num_stop_times", "daily_arrivals", "n_days_schedule_and_rt"]
    ).fmt_number(
        columns = ["daily_service_hours"], decimals=1
    ).tab_spanner(
        label="Schedule",
        columns = schedule_cols
    ).tab_header(
        title = "Schedule + RT Summary Metrics", 
        subtitle = f"{one_month_formatted}"
    )
)

chart_utils.format_great_table(schedule_table, day_type_grouping = False).tab_stub(
    groupname_col="tu_name", rowname_col = "day_type")

## General RT Metrics

<span style="color:#4477aa">**Update Availability Goal 1:** 2+ vehicle positions or trip updates messages per minute.</span>

<span style="color:#4477aa">**Update Availability Goal 2:** 100% routes are covered by RT, and 75%+ of trips have RT.</span>

In [ ]:
rt_table1 = (
    GT(df[["schedule_name"] + operator_cols + ["n_trips"] + tu_cols])
    .cols_label(
        schedule_name = "Schedule",
        n_trips = "# Scheduled Trips",
        n_tu_trips = "# Trips (Trip Updates)",
        tu_messages_per_minute = "Messages per Minute",
        pct_tu_trips = "% Trips",
        pct_tu_routes = "% Routes",
    ).fmt_number(
        columns = ["tu_messages_per_minute"], decimals=1
    ).fmt_percent(
        columns=["pct_tu_trips", "pct_tu_routes"], decimals=1
    ).fmt_integer(columns = ["n_trips", "n_tu_trips"]
    ).tab_stub(rowname_col="day_type", groupname_col="tu_name")
    .cols_hide(columns=["tu_name"])
)

chart_utils.format_great_table(rt_table1, day_type_grouping = False).pipe(
    gte.gt_color_box, 
    columns=["tu_messages_per_minute"], 
    palette="Blues",
    domain=[1, 3]
).pipe(
    gte.gt_hulk_col_numeric, 
    columns=["pct_tu_trips", "pct_tu_routes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")], 
    domain=[0, 1],
    na_color="black", # with opacity at 0.1, this is basically gray
    alpha=0.1
)

In [ ]:
rt_table2 = (
    GT(df[operator_cols + tu_prediction_cols1])
    .cols_label(
        pct_tu_complete_minutes = "% Minutes with 2+ Predictions",
        bus_catch_likelihood = "Bus Catch Likelihood (Early + On-time)",
        p50 = "Prediction Error", 
        n_predictions = "# Predictions",
    ).fmt_percent(columns=["bus_catch_likelihood", "pct_tu_complete_minutes"], decimals=1)
    .fmt_number(columns=["p50"], decimals=1)
    .fmt_integer(columns=["n_predictions"])
    .tab_header(
        title = f"Trip Update Prediction Accuracy Metrics", 
        subtitle = "units are in minutes")
).cols_hide(
    columns=["tu_name"]
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)

rt_table2.pipe(
    gte.gt_hulk_col_numeric, 
    columns=["bus_catch_likelihood", "pct_tu_complete_minutes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")],
    domain=[0, 1],
    alpha=0.1
).pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).cols_move_to_end(columns=["n_predictions"])

Not colored in main report, which is individual feeds.
Here, since all feeds are put together, color them so we can see how they compare.

* Variability (IQR) is ranked across the 3 feeds, so it's more interpretable as a comparison.
* Prediction Spread, prediction padding


In [ ]:
rt_table3 = (
    GT(df[operator_cols + ["p50"] + tu_prediction_cols2])
    .cols_label(
        p50 = "Prediction Error", 
        avg_prediction_spread_minutes = "Prediction Spread / Wobble",
        prediction_padding_minutes = "Prediction Padding",
        iqr = "Variability"
    )
    .fmt_number(columns=["p50", "avg_prediction_spread_minutes", "prediction_padding_minutes"], decimals=1)
    .tab_header(title = f"Trip Update Prediction Accuracy Metrics", 
                subtitle = "units are in minutes")
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)


rt_table3.pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).pipe(
    gte.gt_plt_dumbbell,
    col1='p25',
    col2='p75',
    label = "IQR",
    num_decimals=1,
    col1_color=_color_palette.get_color("valentino"),
    col2_color=_color_palette.get_color("lizard_green"),
    width=100, height=50, # check these each time
    font_size=8
).pipe(
    gte.gt_color_box, 
    columns=["iqr"], 
    palette="YlOrRd"
).pipe(
    gte.gt_color_box,
    columns=["prediction_padding_minutes", "avg_prediction_spread_minutes"],
    palette="BuPu"
)